# One-Stage Exploration (Severstal, Pilot)

## What this notebook is doing now
This notebook is an **exploratory one-stage-only benchmark**.
It sweeps multiple one-stage scoring methods and hyperparameters to identify which candidate is strongest.

## Why this exists
Your FYP is a **one-stage vs two-stage** comparison, but this notebook is currently only for **one-stage exploration**.
Two-stage experiments are run separately in:
- `severstral-osr/notebooks/compare_one_stage_vs_two_stage_small_first.ipynb`

## Caveat
Results here are **exploration outputs**, not the final locked one-stage report yet.
After selecting the best one-stage method from this notebook, we will implement the final one-stage run path in this same notebook.

## Labels used in evaluation
Three-way evaluation:
- `normal` (non-defect)
- `known defect`
- `unknown defect`


In [ ]:
# Cell 1: Repo setup + latest pull
import os, sys, subprocess
from pathlib import Path

REPO = Path('/content/FYP-code')
URL = 'https://github.com/spinelessknave8/FYP_code.git'

if not REPO.exists():
    subprocess.check_call(['git','clone',URL,str(REPO)])
else:
    subprocess.check_call(['git','-C',str(REPO),'fetch','origin'])
    subprocess.check_call(['git','-C',str(REPO),'reset','--hard','origin/main'])

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('cwd:', Path.cwd())
print(subprocess.check_output(['git','-C',str(REPO),'log','--oneline','-n','2'], text=True))


In [ ]:
# Cell 2: Runtime preflight (keep stack stable)
import platform, numpy as np
import torch, torchvision, sklearn

print('python:', platform.python_version())
print('numpy:', np.__version__)
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('sklearn:', sklearn.__version__)

assert np.__version__.startswith('1.26'), (
    'Use numpy 1.26.x for this notebook runtime. '
    'If broken: pip install --force-reinstall numpy==1.26.4 then restart runtime once.'
)


In [ ]:
# Cell 3: Mount Drive + configs + pilot toggles
from google.colab import drive
from pathlib import Path
import yaml, shutil

# ----- toggles -----
HARD_RESET = False
SPLIT = 'b'            # a/b/c/d
SEED = 42

# pilot sizes
NORMAL_TRAIN_MAX = 1200
NORMAL_VAL_MAX = 250
NORMAL_TEST_MAX = 350
KNOWN_TRAIN_PER_CLASS = 180
KNOWN_VAL_PER_CLASS = 60
KNOWN_TEST_PER_CLASS = 100
UNKNOWN_TEST_MAX = 300

# calibration targets
FPR_TARGETS = [0.05, 0.10, 0.20]
MODEL_SELECT_FPR = 0.10
UNKNOWN_FPR_ON_KNOWN_TARGET = 0.10

OUT = Path('/content/drive/MyDrive/fyp_outputs/severstral_one_stage_pilot')

drive.mount('/content/drive')

sev = Path('/content/drive/MyDrive/datasets/severstal')
assert sev.exists(), f'Missing {sev}'
assert (sev/'train.csv').exists(), 'Missing train.csv'
assert (sev/'train_images').exists(), 'Missing train_images'

if HARD_RESET and OUT.exists():
    shutil.rmtree(OUT)
    print('Deleted old output:', OUT)
OUT.mkdir(parents=True, exist_ok=True)

# write minimal colab cfg (for split known class lookup)
base = yaml.safe_load(Path('severstral-osr/configs/default.yaml').read_text())
base['device'] = 'cuda'
base['severstal']['data_root'] = str(sev)
base['severstal']['train_csv'] = 'train.csv'
base['severstal']['images_dir'] = 'train_images'
base['output_dir'] = str(OUT)

split_cfg = yaml.safe_load(Path(f'severstral-osr/configs/split_{SPLIT}.yaml').read_text())
base.update(split_cfg)
cfg_path = Path(f'severstral-osr/configs/split_{SPLIT}.pilot.colab.yaml')
cfg_path.write_text(yaml.safe_dump(base, sort_keys=False))

print('Using split:', SPLIT)
print('Known classes:', base['known_classes'])
print('Output dir:', OUT)


In [ ]:
# Cell 4: Build pilot manifest (normal / known / unknown)
import json, random
from collections import defaultdict
from pathlib import Path

import sys
sys.path.insert(0, str(Path('severstral-osr/src').resolve()))
from data import collect_single_label_defect_samples, collect_normal_image_paths, stratified_split


rng = random.Random(SEED)
known_classes = base['known_classes']
all_classes = [f'Class_{i}' for i in [1,2,3,4]]
unknown_classes = [c for c in all_classes if c not in known_classes]
assert len(unknown_classes)==1, f'Expected 1 unknown class, got {unknown_classes}'
unknown_class = unknown_classes[0]

samples = collect_single_label_defect_samples(str(sev), 'train.csv', 'train_images')
normal_paths = collect_normal_image_paths(str(sev), 'train.csv', 'train_images')

known_samples = [s for s in samples if s[1] in known_classes]
unknown_samples = [s for s in samples if s[1] == unknown_class]

k_train_all, k_val_all, k_test_all = stratified_split(known_samples, train_ratio=0.7, val_ratio=0.15, seed=SEED)

# cap per-class
by_cls_train = defaultdict(list)
by_cls_val = defaultdict(list)
by_cls_test = defaultdict(list)
for p,c in k_train_all: by_cls_train[c].append((p,c))
for p,c in k_val_all: by_cls_val[c].append((p,c))
for p,c in k_test_all: by_cls_test[c].append((p,c))

known_train, known_val, known_test = [], [], []
for c in known_classes:
    rng.shuffle(by_cls_train[c]); rng.shuffle(by_cls_val[c]); rng.shuffle(by_cls_test[c])
    known_train += by_cls_train[c][:KNOWN_TRAIN_PER_CLASS]
    known_val += by_cls_val[c][:KNOWN_VAL_PER_CLASS]
    known_test += by_cls_test[c][:KNOWN_TEST_PER_CLASS]

rng.shuffle(normal_paths)
normal_train = normal_paths[:NORMAL_TRAIN_MAX]
normal_val = normal_paths[NORMAL_TRAIN_MAX:NORMAL_TRAIN_MAX+NORMAL_VAL_MAX]
normal_test = normal_paths[NORMAL_TRAIN_MAX+NORMAL_VAL_MAX:NORMAL_TRAIN_MAX+NORMAL_VAL_MAX+NORMAL_TEST_MAX]

rng.shuffle(unknown_samples)
unknown_test = unknown_samples[:UNKNOWN_TEST_MAX]

manifest = {
    'seed': SEED,
    'split': SPLIT,
    'known_classes': known_classes,
    'unknown_class': unknown_class,
    'normal_train': normal_train,
    'normal_val': normal_val,
    'normal_test': normal_test,
    'known_train': known_train,
    'known_val': known_val,
    'known_test': known_test,
    'unknown_test': unknown_test,
}

manifest_path = OUT / 'pilot_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

print('Manifest:', manifest_path)
print('counts:', {
    'normal_train': len(normal_train), 'normal_val': len(normal_val), 'normal_test': len(normal_test),
    'known_train': len(known_train), 'known_val': len(known_val), 'known_test': len(known_test),
    'unknown_test': len(unknown_test),
})


In [ ]:
# Cell 5: Train small classifier on known classes (for embeddings + logits)
import time, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from src.models.resnet50 import build_resnet50

device = 'cuda' if torch.cuda.is_available() else 'cpu'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TF = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImgDS(Dataset):
    def __init__(self, items, class_to_idx=None):
        self.items = items
        self.class_to_idx = class_to_idx
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p,c = self.items[i]
        x = TF(Image.open(p).convert('RGB'))
        if self.class_to_idx is None:
            return x, -1
        return x, self.class_to_idx[c]

class_to_idx = {c:i for i,c in enumerate(manifest['known_classes'])}

train_ds = ImgDS(manifest['known_train'], class_to_idx)
val_ds = ImgDS(manifest['known_val'], class_to_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

model = build_resnet50(num_classes=len(class_to_idx), pretrained=True).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()

EPOCHS = 6
best_val = -1.0
best_state = None

t0 = time.time()
for ep in range(EPOCHS):
    model.train(); tr_c=0; tr_n=0
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        opt.zero_grad(); logits=model(x); loss=crit(logits,y); loss.backward(); opt.step()
        tr_c += (logits.argmax(1)==y).sum().item(); tr_n += x.size(0)

    model.eval(); va_c=0; va_n=0
    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(device), y.to(device)
            logits = model(x)
            va_c += (logits.argmax(1)==y).sum().item(); va_n += x.size(0)
    tr_acc = tr_c / max(1,tr_n)
    va_acc = va_c / max(1,va_n)
    print(f'epoch {ep+1}/{EPOCHS} train_acc={tr_acc:.3f} val_acc={va_acc:.3f}')
    if va_acc > best_val:
        best_val = va_acc
        best_state = {k:v.cpu() for k,v in model.state_dict().items()}

model.load_state_dict(best_state)
train_sec = time.time()-t0

clf_dir = OUT / 'classifier'
clf_dir.mkdir(parents=True, exist_ok=True)
torch.save({'model_state':model.state_dict(), 'class_to_idx':class_to_idx}, clf_dir/'classifier.pt')
(clf_dir/'train_metrics.json').write_text(json.dumps({'best_val_acc':float(best_val),'train_sec':train_sec}, indent=2))
print('Saved classifier:', clf_dir/'classifier.pt')
print('train_sec:', round(train_sec,1), 'best_val_acc:', round(best_val,4))


In [ ]:
# Cell 6: Extract embeddings/logits for all groups
import json, time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from src.models.resnet50 import build_resnet50
from src.models.embedding import EmbeddingExtractor

device = 'cuda' if torch.cuda.is_available() else 'cpu'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TF = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class GroupDS(Dataset):
    def __init__(self, items, label_mode='known'):
        self.items = items
        self.label_mode = label_mode
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p,c = self.items[i]
        x = TF(Image.open(p).convert('RGB'))
        return x, c

ckpt = torch.load(OUT/'classifier'/'classifier.pt', map_location='cpu')
class_to_idx = ckpt['class_to_idx']
idx_to_class = {v:k for k,v in class_to_idx.items()}

model = build_resnet50(num_classes=len(class_to_idx), pretrained=False)
model.load_state_dict(ckpt['model_state'])
model.to(device).eval()
extractor = EmbeddingExtractor(model).to(device)

def pack_items(group_name):
    if group_name.startswith('normal'):
        return [(p,'normal') for p in manifest[group_name]]
    return manifest[group_name]

def run_group(name):
    items = pack_items(name)
    ds = GroupDS(items)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    all_emb, all_logits, all_raw = [], [], []
    with torch.no_grad():
        for x,raw in loader:
            x = x.to(device)
            logits = model(x)
            emb = extractor(x)
            all_emb.append(emb.cpu().numpy())
            all_logits.append(logits.cpu().numpy())
            all_raw += list(raw)
    emb = np.concatenate(all_emb)
    logits = np.concatenate(all_logits)
    return {'emb':emb, 'logits':logits, 'raw_labels':all_raw}

groups = ['normal_train','normal_val','normal_test','known_train','known_val','known_test','unknown_test']
cache = {}
for g in groups:
    cache[g] = run_group(g)
    print(g, cache[g]['emb'].shape, cache[g]['logits'].shape)

# numeric labels for known groups
def encode_known(raw):
    return np.array([class_to_idx[r] for r in raw], dtype=np.int64)

cache['known_train']['y'] = encode_known(cache['known_train']['raw_labels'])
cache['known_val']['y'] = encode_known(cache['known_val']['raw_labels'])
cache['known_test']['y'] = encode_known(cache['known_test']['raw_labels'])

npz_path = OUT / 'pilot_embeddings.npz'
np.savez(
    npz_path,
    normal_train_emb=cache['normal_train']['emb'], normal_val_emb=cache['normal_val']['emb'], normal_test_emb=cache['normal_test']['emb'],
    known_train_emb=cache['known_train']['emb'], known_val_emb=cache['known_val']['emb'], known_test_emb=cache['known_test']['emb'], unknown_test_emb=cache['unknown_test']['emb'],
    known_train_y=cache['known_train']['y'], known_val_y=cache['known_val']['y'], known_test_y=cache['known_test']['y'],
    known_test_logits=cache['known_test']['logits'], unknown_test_logits=cache['unknown_test']['logits'], normal_test_logits=cache['normal_test']['logits'],
    known_val_logits=cache['known_val']['logits'], normal_val_logits=cache['normal_val']['logits']
)
print('Saved:', npz_path)


In [ ]:
# Cell 7: One-stage scorer sweep + hyperparameter tuning
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from scipy.special import logsumexp

from src.osr.gaussian_mahalanobis import fit_gaussians, batch_min_mahalanobis

arr = np.load(OUT / 'pilot_embeddings.npz', allow_pickle=True)

NTR = arr['normal_train_emb']; NVAL = arr['normal_val_emb']; NTEST = arr['normal_test_emb']
KTR = arr['known_train_emb'];  KVAL = arr['known_val_emb'];  KTEST = arr['known_test_emb']; UTEST = arr['unknown_test_emb']
y_ktr = arr['known_train_y']; y_kval = arr['known_val_y']; y_ktest = arr['known_test_y']
log_kval = arr['known_val_logits']; log_ktest = arr['known_test_logits']; log_utest = arr['unknown_test_logits']; log_nval = arr['normal_val_logits']; log_ntest = arr['normal_test_logits']

# Shared unknown scorer + known-class predictor (class-conditional mahal on known-train)
unknown_cov_reg = 1e-3
known_params = fit_gaussians(KTR, y_ktr, unknown_cov_reg)
U_VAL = batch_min_mahalanobis(KVAL, known_params)
U_TEST_K = batch_min_mahalanobis(KTEST, known_params)
U_TEST_U = batch_min_mahalanobis(UTEST, known_params)

# unknown threshold from known-val only (ID-only calibration)
u_tau = float(np.quantile(U_VAL, 1.0 - UNKNOWN_FPR_ON_KNOWN_TARGET))

def predict_known_label_mahal(X):
    cls = sorted(list(known_params.keys()))
    d = np.stack([np.sum((X - known_params[c]['mu']) @ known_params[c]['inv_cov'] * (X - known_params[c]['mu']), axis=1) for c in cls], axis=1)
    return np.array([cls[i] for i in np.argmin(d, axis=1)], dtype=np.int64)

known_pred_labels = predict_known_label_mahal(KTEST)

# 3-way truth: 0 normal, 1 known defect, 2 unknown defect
y3_true = np.concatenate([
    np.zeros(len(NTEST), dtype=np.int64),
    np.ones(len(KTEST), dtype=np.int64),
    np.full(len(UTEST), 2, dtype=np.int64),
])

def tau_from_fpr(normal_scores, fpr_target):
    return float(np.quantile(normal_scores, 1.0 - fpr_target))

def eval_screening(S_n, S_k, S_u, tau):
    y_true = np.concatenate([np.zeros(len(S_n)), np.ones(len(S_k)+len(S_u))])
    y_pred = np.concatenate([(S_n > tau).astype(int), (S_k > tau).astype(int), (S_u > tau).astype(int)])
    fpr_normal = float(np.mean(S_n > tau))
    tpr_defect = float(np.mean(np.concatenate([S_k,S_u]) > tau))
    return y_true, y_pred, fpr_normal, tpr_defect

def eval_three_way(S_n, S_k, S_u, tau, U_k, U_u, u_tau, known_cls_pred):
    pred_n = np.where(S_n > tau, 1, 0)  # 0 normal, else defect branch
    # for normals predicted defect -> known/unknown by unknown scorer not available; map to known(1)
    y_n = np.where(pred_n==0, 0, 1)

    # known defects
    known_is_def = S_k > tau
    known_is_unknown = U_k > u_tau
    y_k = np.where(~known_is_def, 0, np.where(known_is_unknown, 2, 1))

    # unknown defects
    unk_is_def = S_u > tau
    unk_is_unknown = U_u > u_tau
    y_u = np.where(~unk_is_def, 0, np.where(unk_is_unknown, 2, 1))

    y3_pred = np.concatenate([y_n, y_k, y_u])

    acc = float(accuracy_score(y3_true, y3_pred))
    prec, rec, f1, _ = precision_recall_fscore_support(y3_true, y3_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y3_true, y3_pred, labels=[0,1,2]).tolist()

    tpr_unknown_within_defect = float(np.mean(y_u == 2))
    fpr_known_as_unknown = float(np.mean(y_k == 2))

    return {
        'acc_3way': acc,
        'macro_precision_3way': float(prec),
        'macro_recall_3way': float(rec),
        'macro_f1_3way': float(f1),
        'tpr_unknown_within_defect': tpr_unknown_within_defect,
        'fpr_known_as_unknown': fpr_known_as_unknown,
        'cm_3way': cm,
    }


def screening_from_method(method, params):
    if method == 'global_mahalanobis':
        reg = float(params['cov_reg'])
        mu = NTR.mean(axis=0)
        cov = np.cov(NTR, rowvar=False) + reg*np.eye(NTR.shape[1])
        inv = np.linalg.inv(cov)
        def m(X):
            D = X - mu
            return np.sum((D @ inv) * D, axis=1)
        return m(NVAL), m(NTEST), m(KVAL), m(KTEST), m(UTEST)

    if method == 'global_knn':
        k = int(params['k'])
        nn = NearestNeighbors(n_neighbors=k, metric='euclidean').fit(NTR)
        def s(X):
            d,_ = nn.kneighbors(X)
            return d.mean(axis=1)
        return s(NVAL), s(NTEST), s(KVAL), s(KTEST), s(UTEST)

    if method == 'one_class_svm_rbf':
        m = OneClassSVM(kernel='rbf', nu=float(params['nu']), gamma=params['gamma'])
        m.fit(NTR)
        def s(X):
            return -m.decision_function(X)  # higher => more anomalous
        return s(NVAL), s(NTEST), s(KVAL), s(KTEST), s(UTEST)

    if method == 'isolation_forest':
        m = IsolationForest(
            n_estimators=int(params['n_estimators']),
            max_samples=params['max_samples'],
            contamination=float(params['contamination']),
            random_state=SEED,
            n_jobs=-1,
        )
        m.fit(NTR)
        def s(X):
            return -m.score_samples(X)  # higher => more anomalous
        return s(NVAL), s(NTEST), s(KVAL), s(KTEST), s(UTEST)

    if method == 'energy_score':
        T = float(params['temperature'])
        def energy(logits):
            return -T * logsumexp(logits / T, axis=1)
        S_n_val = energy(log_nval)
        S_n_test = energy(log_ntest)
        S_k_val = energy(log_kval)
        S_k_test = energy(log_ktest)
        S_u_test = energy(log_utest)
        return S_n_val, S_n_test, S_k_val, S_k_test, S_u_test

    if method == 'class_conditional_mahalanobis':
        reg = float(params['cov_reg'])
        p = fit_gaussians(KTR, y_ktr, reg)
        S_n_val = batch_min_mahalanobis(NVAL, p)
        S_n_test = batch_min_mahalanobis(NTEST, p)
        S_k_val = batch_min_mahalanobis(KVAL, p)
        S_k_test = batch_min_mahalanobis(KTEST, p)
        S_u_test = batch_min_mahalanobis(UTEST, p)
        return S_n_val, S_n_test, S_k_val, S_k_test, S_u_test

    raise ValueError(method)

search_space = {
    'global_mahalanobis': [
        {'cov_reg': 1e-4}, {'cov_reg': 1e-3}, {'cov_reg': 1e-2}
    ],
    'global_knn': [
        {'k': 3}, {'k': 5}, {'k': 11}, {'k': 21}
    ],
    'one_class_svm_rbf': [
        {'nu': 0.01, 'gamma': 'scale'}, {'nu': 0.05, 'gamma': 'scale'},
        {'nu': 0.10, 'gamma': 'scale'}, {'nu': 0.05, 'gamma': 0.01}, {'nu': 0.05, 'gamma': 0.001}
    ],
    'isolation_forest': [
        {'n_estimators': 100, 'max_samples': 'auto', 'contamination': 0.05},
        {'n_estimators': 300, 'max_samples': 'auto', 'contamination': 0.05},
        {'n_estimators': 300, 'max_samples': 0.5, 'contamination': 0.10},
    ],
    'energy_score': [
        {'temperature': 1.0}, {'temperature': 2.0}, {'temperature': 5.0}
    ],
    'class_conditional_mahalanobis': [
        {'cov_reg': 1e-4}, {'cov_reg': 1e-3}, {'cov_reg': 1e-2}
    ],
}

rows = []
for method, plist in search_space.items():
    for p in plist:
        S_n_val, S_n_test, S_k_val, S_k_test, S_u_test = screening_from_method(method, p)

        # model selection on val at fixed FPR (ID-only calibration)
        tau_sel = tau_from_fpr(S_n_val, MODEL_SELECT_FPR)
        _, _, fpr_sel, tpr_sel = eval_screening(S_n_val, S_k_val, S_k_val[:0], tau_sel)

        # screening AUROC on test (normal vs any defect)
        y_bin = np.concatenate([np.zeros(len(S_n_test)), np.ones(len(S_k_test)+len(S_u_test))])
        s_bin = np.concatenate([S_n_test, S_k_test, S_u_test])
        auroc = float(roc_auc_score(y_bin, s_bin)) if len(np.unique(y_bin))>1 else float('nan')

        for fpr_t in FPR_TARGETS:
            tau = tau_from_fpr(S_n_val, fpr_t)
            _, _, fpr_normal, tpr_defect = eval_screening(S_n_test, S_k_test, S_u_test, tau)
            m3 = eval_three_way(S_n_test, S_k_test, S_u_test, tau, U_TEST_K, U_TEST_U, u_tau, known_pred_labels)
            rows.append({
                'method': method,
                **{f'param_{k}':v for k,v in p.items()},
                'model_select_tau@val_fpr10': tau_sel,
                'val_tpr_defect@fpr10': tpr_sel,
                'screening_auroc_test': auroc,
                'fpr_target': fpr_t,
                'tau': tau,
                'fpr_normal': fpr_normal,
                'tpr_defect': tpr_defect,
                **m3,
            })

res = pd.DataFrame(rows)
res.to_csv(OUT / 'one_stage_scorer_sweep.csv', index=False)
print('Wrote:', OUT / 'one_stage_scorer_sweep.csv', 'rows=', len(res))

# pick one best per method under fpr_target==0.10 by max macro_f1_3way then tpr_defect
best = (
    res[res['fpr_target']==0.10]
    .sort_values(['method','macro_f1_3way','tpr_defect'], ascending=[True,False,False])
    .groupby('method', as_index=False)
    .head(1)
    .sort_values('macro_f1_3way', ascending=False)
)
best.to_csv(OUT / 'one_stage_best_per_method.csv', index=False)
print('Wrote:', OUT / 'one_stage_best_per_method.csv')
display(best[[
    'method','fpr_target','screening_auroc_test','tpr_defect','fpr_normal','tpr_unknown_within_defect','fpr_known_as_unknown','acc_3way','macro_f1_3way'
]])


In [ ]:
# Cell 8: Plots + quick summary
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

res = pd.read_csv(OUT / 'one_stage_scorer_sweep.csv')
best = pd.read_csv(OUT / 'one_stage_best_per_method.csv')

# Plot 1: AUROC by method (from best table)
plt.figure(figsize=(9,4))
plt.bar(best['method'], best['screening_auroc_test'])
plt.xticks(rotation=25, ha='right')
plt.ylabel('AUROC (normal vs defect)')
plt.title('One-stage: best config per method')
plt.tight_layout()
plt.savefig(OUT / 'plot_auroc_best_methods.png', dpi=160)
plt.show()

# Plot 2: TPR/FPR trade points for each method
fig, ax = plt.subplots(figsize=(7,5))
for m, g in res.groupby('method'):
    g = g.sort_values('fpr_normal')
    ax.plot(g['fpr_normal'], g['tpr_defect'], marker='o', label=m, alpha=0.8)
ax.set_xlabel('FPR normal')
ax.set_ylabel('TPR defect')
ax.set_title('Operating points across sweeps')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(OUT / 'plot_tpr_vs_fpr_sweeps.png', dpi=160)
plt.show()

print('Saved plots in:', OUT)
print('
Best per method (fpr target=0.10):')
display(best[[
    'method','screening_auroc_test','tpr_defect','fpr_normal','tpr_unknown_within_defect','fpr_known_as_unknown','acc_3way','macro_f1_3way'
]])


## Rerun Guide
1. Re-run Cell 1 to sync latest repo.
2. Re-run Cell 3 (`HARD_RESET=True` if you need a clean rerun).
3. Run Cells 4–8 top to bottom.

Outputs:
- `one_stage_scorer_sweep.csv`
- `one_stage_best_per_method.csv`
- `plot_auroc_best_methods.png`
- `plot_tpr_vs_fpr_sweeps.png`
